# 自动微分绘制 sin(x) 及其导数

目标：令 `f(x) = sin(x)`，绘制 `f(x)` 和 `df(x)/dx` 的图像。其中导数不直接使用 `f'(x) = cos(x)`，而是让 PyTorch 根据计算图自动求出来。

## `torch.autograd.grad` 的含义

这一句是在计算 `f_x = sin(x)` 关于 `x` 的导数：

```python
df_dx = torch.autograd.grad(
    outputs=f_x,
    inputs=x,
    grad_outputs=torch.ones_like(f_x)
)[0]
```

- `outputs=f_x`：表示要对哪个结果求导，也就是对 `f_x` 求导。
- `inputs=x`：表示关于哪个变量求导，也就是求 `f_x` 对 `x` 的导数。
- `grad_outputs=torch.ones_like(f_x)`：因为 `f_x` 是一个向量，不是单个标量，所以要告诉 PyTorch 每个输出分量的上游梯度是多少。这里全是 `1`，等价于对 `f_x.sum()` 关于 `x` 求导。
- `[0]`：`torch.autograd.grad(...)` 返回的是元组，这里只对一个输入 `x` 求导，所以取第一个结果就是 `df_dx`。

直观理解：`x` 里有 200 个点，`f_x` 里也有 200 个 `sin(x)` 的值。自动微分会得到每个点上 `sin(x)` 对对应 `x` 的局部导数。

In [ ]:
import math
import torch
import matplotlib.pyplot as plt

# 构造一组 x 点，并让 PyTorch 跟踪它们的梯度。
x = torch.linspace(-2 * math.pi, 2 * math.pi, 200, requires_grad=True)

# 只定义 f(x)=sin(x)，这里不手写 cos(x)。
f_x = torch.sin(x)

# outputs=f_x：要对 f_x 求导。
# inputs=x：求 f_x 关于 x 的导数。
# grad_outputs 全是 1，等价于对 f_x.sum() 关于 x 求导。
# 返回值是元组，所以用 [0] 取出关于 x 的梯度。
df_dx = torch.autograd.grad(
    outputs=f_x,
    inputs=x,
    grad_outputs=torch.ones_like(f_x)
)[0]

# 画图时要 detach()，避免 matplotlib 直接处理带梯度追踪的张量。
x_np = x.detach().numpy()
f_np = f_x.detach().numpy()
df_np = df_dx.detach().numpy()

plt.figure(figsize=(7, 4))
plt.plot(x_np, f_np, label='f(x) = sin(x)')
plt.plot(x_np, df_np, label='df(x)/dx by autograd')
plt.xlabel('x')
plt.ylabel('value')
plt.grid(True)
plt.legend()
plt.show()